Looking at the extensibility of the models to unseen chemicals

In [121]:
import ast

#import block

#imports
import numpy as np
import ast
import matplotlib.pyplot as plt
import shutil
import sys
import os.path
import json
import seaborn as sns
import pandas as pd
from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem import MACCSkeys
from rdkit.Chem.AtomPairs import Pairs
from rdkit.Chem.rdFingerprintGenerator import GetRDKitFPGenerator
import ot
import csv
from pickle import dump
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import pairwise_distances
from scipy.optimize import minimize
from scipy.stats import bernoulli
from scipy.special import expit as sigmoid
from sklearn.datasets import make_moons
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, RationalQuadratic
from multiprocessing import Pool

#import EMD calculation from Jiale's project
from Polymer_Similarity import EMD_Calculation_pot as EMD

In [159]:
# function definition block
def str_to_list(string):
    #string='"'+str(string)+'"'
    counter=string.count(',')
    #print(counter)
    list=[]
    for i in range(counter):
        if counter ==1:
            string=string[1:len(string)-1]
        if i==0 and counter>1:
            j=string.index(',')
            list+=[string[2:j-1]]
            string=string[j+1:len(string)-1]
            if string[0]==' ':
                string=string[1:]
        elif i==counter-1:
            j=string.index(',')
            list+=[string[1:j-1]]
            if string[j+1]==' ':
                list+=[string[j+3:len(string)-1]]
            else:
                list+=[string[j+2:len(string)-1]]
        else:
            j=string.index(',')
            list+=[string[1:j-1]]
            if string[0]==' ':
                string=string[j+2:]
            else:
                string=string[j+1:]
        #print(string)

    return list

def concatenate_indices(indices,leave_out):
    #indices is a list of the split arrays, leave out is the index number of the test set
    newlist=[]
    test=None
    for i in range(len(indices)):
        if i==leave_out:
            test=indices[i]
        else:
            newlist+=[indices[i]]
    train=newlist[0]
    for i in range(1,len(newlist)):
        train=np.concatenate((train,newlist[i]),0)
    return [train,test]

def rf_cross_val(C, xs, ys,testx,testy,name):
    #C should be [n_estimators, max_depth,max_leaf_nnodes]
    #xs and ys give a list of the different dataset splices - must be same length, indices in each must match
        #this should be the full training set (will iterate which set is being used as the validation set for cross validation
    #added in option for testx and test y to be list of tests rather than just one test
    scores = []
    for i in range(0, len(xs)):
        X = concatenate_indices(xs, i)
        Y = concatenate_indices(ys, i)
        model = RandomForestClassifier(n_estimators=C[0], max_depth=C[1], min_samples_split=2, min_samples_leaf=1,max_leaf_nodes=C[2],random_state=42)
        model.fit(X[0], Y[0])
        scores += [model.score(X[1], Y[1])]
    score_array = np.array(scores)
    mean = np.mean(score_array)
    std = np.std(score_array)

    #now evaluate against unseen testing data
    X = concatenate_indices(xs, None)
    Y = concatenate_indices(ys, None)
    model = RandomForestClassifier(n_estimators=C[0], max_depth=C[1], min_samples_split=2, min_samples_leaf=1,max_leaf_nodes=C[2],random_state=42)
    model.fit(X[0], Y[0])
    if isinstance(testx,list):
        eval=[]
        for i in range(len(testx)):
            eval+=[model.score(testx[i],testy[i])]
    else:
        eval=model.score(testx,testy)
    with open("./Updated_Modeling_Rev/"+str(name)+"RF_[n_estim,max_depth,max_leaf]_"+str(C)+".pkl", "wb") as f:
        dump(model, f, protocol=5)
    return [mean, std, eval]

def train_rf(C,xs,ys,name):
    #C should be [n_estimators, max_depth,max_leaf_nnodes]
    #xs and ys should be list of the training data (no validation set, test set should be left out)
    # X = concatenate_indices(xs, None)
    # Y = concatenate_indices(ys, None)
    X=xs
    print('X',np.shape(X))
    # print('Y',np.shape(Y))
    Y=ys
    print('Y',np.shape(Y))
    model = RandomForestClassifier(n_estimators=C[0], max_depth=C[1], min_samples_split=2, min_samples_leaf=1, max_leaf_nodes=C[2],random_state=42)
    model.fit(X, Y)
    with open("./Chemical_extensibility/"+str(name)+"_RF_[n_estim,max_depth,max_leaf]_"+str(C)+".pkl", "wb") as f:
        dump(model, f, protocol=5)
    return model

def update_rf(old_model,C,xs,ys):
    #this takes an existing trained rf and fits additional trees given additional training data
    #C should be the number of additional estimators (trees) to fit
    #xs and ys should be list of the training data (no validation set, test set should be left out)
    n_estims=len(old_model.estimators_)

    X = concatenate_indices(xs, None)
    Y = concatenate_indices(ys, None)
    model_list=[]
    estimator_amounts=[]
    model = old_model.set_params(n_estimators=C+n_estims,warm_start=True)
    model.fit(X[0], Y[0])
    return model


def lin_cross_val(C, xs, ys,xtest,ytest,name):
    #C should be [loss, alpha]
    #xs and ys give a list of the different dataset splices - must be same length, indices in each must match
        #this should be the full training set (will iterate which set is being used as the validation set for cross validation
    scores = []
    for i in range(0, len(xs)):
        X = concatenate_indices(xs, i)
        Y = concatenate_indices(ys, i)
        model =SGDClassifier(loss=C[0], alpha=C[1],shuffle=True)
        model.fit(X[0], Y[0])
        scores += [model.score(X[1], Y[1])]
    score_array = np.array(scores)
    mean = np.mean(score_array)
    std = np.std(score_array)
    X = concatenate_indices(xs, None)
    Y = concatenate_indices(ys, None)
    model = SGDClassifier(loss=C[0], alpha=C[1], shuffle=True)
    model.fit(X[0],Y[0])
    eval=model.score(xtest,ytest)
    #save the model
    with open("./240527_with_Terpolymers_Optimization Models/SGD_"+str(name)+"_[loss,alpha]_"+str(C)+".pkl", "wb") as f:
        dump(model, f, protocol=5)
    return [mean, std,eval]

def train_lin(C, xs,ys,name):
    X=xs
    print('X',np.shape(X))
    # print('Y',np.shape(Y))
    Y=ys
    print('Y',np.shape(Y))
    model = SGDClassifier(loss=C[0], alpha=C[1], shuffle=True,random_state=42)
    model.fit(X, Y)
    with open("./Updated_Modeling_Rev/"+str(name)+"SDG_"+str(name)+"[loss,alpha]"+str(C)+".pkl", "wb") as f:
        dump(model, f, protocol=5)
    return model

def update_lin(old_model, xs, ys):
    #xs are additional new data that needs to be fit
    #ys are labels corresponding to xs
    class_biodeg=[0,1] #pretty sure since we use 0 or 1 this is all we've got
    model=old_model.partial_fit(xs,ys,classes=class_biodeg)
    return model

def get_EMD_array(sets,basis,EMD_values=["RDKFingerprint","Tanimoto",5,512]):
    #sets = list of sets (pd dataframes) that need EMD calculations against the basis
    #if basis=None then the concatenation of the sets is used as basis. If not, give polymer basis
    #EMD_values gives [embedding_function, similarity_score_function, radius, num_bits]

    EMD_arrays=[]
    for k in range(len(sets)):
        col = []
        for i in range(len(sets[k].index)):
            row = []
            for j in range(len(basis.index)):
                X = sets[k]['SMILES LIST'][i]
                #print(X)
                if isinstance(X,str):
                    X = str_to_list(X)
                #print(X)
                Y = sets[k]['Ratio List'][i]
                #print(basis['SMILES_List'])
                #print("basis",basis['SMILES LIST'][j])
                Z = basis['SMILES LIST'][j]
                #print(Z)
                if isinstance(Z,str):
                    Z = str_to_list(Z)
                #print(Z)
                W = basis['Ratio List'][j]
                # W=str_to_list(W)
                #print(X, Y, Z, W)
                score = EMD(
                    query_smiles_list=X,
                    query_smiles_weight_list=Y,
                    target_smiles_list=Z,
                    target_smiles_weight_list=W,
                    embedding_function=EMD_values[0],
                    similarity_score_function=EMD_values[1],
                    radius=EMD_values[2],
                    num_bits=EMD_values[3],
                )
                row += [score]
            col += [row]
        # save the list of lists as a npy array
        formed = np.array(col)
        EMD_arrays+=[formed]
    return EMD_arrays


In [26]:
#make a set of identical splits (which polymers are in which split) with updated details.
ter_Julia_r=pd.read_csv('./Library_Data_Julia_Consolidated_Revision.csv',sep='\t',header=0)
# print(ter_Julia_r)
ter_other_r=pd.read_csv('./Library_Data_Consolidated_Revision.csv',header=0)
# print(ter_other_r)
ter_Julia=ter_Julia_r[['Polymer_Name_P','Biodegradable','Ratio 1','Ratio 2','Ratio 3']]
ter_other=ter_other_r[['Polymer_Name_P','Biodegradable','Ratio 1','Ratio 2','Ratio 3']]
ter_all=pd.concat([ter_Julia,ter_other])
print(ter_all)
for i in range(6):
    all=pd.read_csv('./Terpolymer_Splits/split_'+str(i)+'.csv',header=0)
    set_A=pd.read_csv('./Terpolymer_Splits/split_'+str(i)+'_A'+'.csv',header=0)
    set_B=pd.read_csv('./Terpolymer_Splits/split_'+str(i)+'_B'+'.csv',header=0)
    new_all=ter_all[ter_all['Polymer_Name_P'].isin(all['Polymer_Name_P'])]
    new_setA=ter_all[ter_all['Polymer_Name_P'].isin(set_A['Polymer_Name_P'])]
    new_setB=ter_all[ter_all['Polymer_Name_P'].isin(set_B['Polymer_Name_P'])]
    new_all.to_csv(path_or_buf='./Terpolymer_Splits_Rev/split_'+str(i)+'.csv')
    new_setA.to_csv(path_or_buf='./Terpolymer_Splits_Rev/split_'+str(i)+'_A'+'.csv')
    new_setB.to_csv(path_or_buf='./Terpolymer_Splits_Rev/split_'+str(i)+'_B'+'.csv')

          Polymer_Name_P Biodegradable  Ratio 1  Ratio 2  Ratio 3
0    ADA25_DMI25_BND50_P             Y      1.4      0.6      2.0
1    ADA25_DMI25_BTD50_P             Y      0.9      1.1      2.0
2    ADA25_DMT25_BPA50_P             N      1.8      0.2      2.0
3    ADA25_DMT25_DPD50_P             N      1.1      0.9      2.0
4    ADA25_DMT25_HND50_P             Y      1.2      0.8      2.0
..                   ...           ...      ...      ...      ...
230  DGA25_CCD25_GGE50_P             Y      1.0      1.0      2.0
231  EBA25_AZA25_BPA50_P             Y      0.9      1.1      2.0
232  GLA25_SBA25_BPA50_P             Y      1.1      0.9      2.0
233  EBA25_SBA25_BPA50_P             Y      0.9      1.1      2.0
234  GLA25_SUA25_BPA50_P             N      1.7      0.3      2.0

[300 rows x 5 columns]


In [27]:
#write npy file of the ys for each of the ys for each set

for i in range(6):
    all=pd.read_csv('./Terpolymer_Splits_Rev/split_'+str(i)+'.csv',header=0)
    set_A=pd.read_csv('./Terpolymer_Splits_Rev/split_'+str(i)+'_A'+'.csv',header=0)
    set_B=pd.read_csv('./Terpolymer_Splits_Rev/split_'+str(i)+'_B'+'.csv',header=0)
    all_biodeg=all['Biodegradable'].replace('N',0)
    all_biodeg=all_biodeg.replace('Y',1)
    set_A_biodeg=set_A['Biodegradable'].replace('N',0)
    set_A_biodeg=set_A_biodeg.replace('Y',1)
    set_B_biodeg=set_B['Biodegradable'].replace('Y',1)
    set_B_biodeg=set_B_biodeg.replace('N',0)
    all_biodeg_np=all_biodeg.to_numpy()
    set_A_biodeg_np=set_A_biodeg.to_numpy()
    set_B_biodeg_np=set_B_biodeg.to_numpy()
    print(set_A_biodeg_np)
    np.save('./Terpolymer_Splits_Rev/split'+str(i)+'_ys',all_biodeg_np,allow_pickle=False)
    np.save('./Terpolymer_Splits_Rev/split'+str(i)+'_A_ys',set_A_biodeg_np,allow_pickle=False)
    np.save('./Terpolymer_Splits_Rev/split'+str(i)+'_B_ys',set_B_biodeg_np,allow_pickle=False)

[0 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
[1 0 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
[1 0 0 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1]
[1 1 1 0 1 0 0 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
[1 1 1 1 1 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
[0 0 1 0 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1]


/var/folders/t5/wsvlnfc52kd7dmby8bpwq66r0000gn/T/ipykernel_72556/4224661599.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  all_biodeg=all_biodeg.replace('Y',1)
/var/folders/t5/wsvlnfc52kd7dmby8bpwq66r0000gn/T/ipykernel_72556/4224661599.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  set_A_biodeg=set_A_biodeg.replace('Y',1)
/var/folders/t5/wsvlnfc52kd7dmby8bpwq66r0000gn/T/ipykernel_72556/4224661599.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the 

In [109]:
#load original and reduced data
os0=pd.read_csv('./ML_splits/original_split0.csv',header=0)
os1=pd.read_csv('./ML_splits/original_split1.csv',header=0)
os2=pd.read_csv('./ML_splits/original_split2.csv',header=0)
os3=pd.read_csv('./ML_splits/original_split3.csv',header=0)
os4=pd.read_csv('./ML_splits/original_split4.csv',header=0)
os5=pd.read_csv('./ML_splits/original_split5.csv',header=0)
os_train=[os0,os1,os2,os3,os4, os5] #all the original data
os_train_pd=pd.concat(os_train).reset_index(drop=True)
rl0=pd.read_csv('./Reduced_ML_Splits/reduced_split0.csv',header=0)
rl1=pd.read_csv('./Reduced_ML_Splits/reduced_split1.csv',header=0)
rl2=pd.read_csv('./Reduced_ML_Splits/reduced_split2.csv',header=0)
rl3=pd.read_csv('./Reduced_ML_Splits/reduced_split3.csv',header=0)
rl4=pd.read_csv('./Reduced_ML_Splits/reduced_split4.csv',header=0)
rl5=pd.read_csv('./Reduced_ML_Splits/reduced_split5.csv',header=0)
rl_train=[rl0,rl1,rl2,rl3,rl4,rl5] #all the homopolymer data
rl_train_pd=pd.concat(rl_train).reset_index(drop=True)
print(os_train_pd)
print(os0)

     Unnamed: 0     Name                                        SMILES_List  \
0             2   AC_101             ['C(=O)c(cc1)ccc1C(=O)', 'OCCCCCCCCO']   
1            16   AC_118  ['C(=O)c(cc1)ccc1C(=O)', 'Oc1c(C)cc(cc1(C))C(C...   
2            36   AC_171  ['C(=O)C(=O)', 'Oc1ccc(cc1)C(C)(c2ccccc2)c3ccc...   
3            38   AC_174                         ['C(=O)CC(=O)', 'OCC=CCO']   
4            47   AC_196                     ['C(=O)C(=O)', 'OCCCCCCCCCCO']   
..          ...      ...                                                ...   
651         637   ROP_87             ['C(=O)C=Cc(cccc1)c1O', 'C(=O)CCCCCO']   
652         638   ROP_89              ['C(=O)C=Cc(cccc1)c1O', 'C(=O)CCCCO']   
653         641   ROP_92              ['C(=O)CCCCO', 'C(=O)C=Cc(cccc1)c1O']   
654         644   ROP_95          ['C(=O)CCCCC(CCCC)O', 'C(=O)CCCC(CCCC)O']   
655         654  ROPm_26                   ['C(=O)COC(=O)CO', 'C(=O)CCCCO']   

    Ratio_List  Biodegradability  
0       [1, 1]  

In [148]:
#write cleaned up dataframe with original data
orig_dataframe = pd.DataFrame({'SMILES LIST': os_train_pd['SMILES_List'], 'Ratio List': os_train_pd['Ratio_List'],'Biodegradability': os_train_pd['Biodegradability']})
orig_dataframe['SMILES LIST']=orig_dataframe['SMILES LIST'].apply(ast.literal_eval)
print(orig_dataframe)

                                           SMILES LIST Ratio List  \
0                   [C(=O)c(cc1)ccc1C(=O), OCCCCCCCCO]     [1, 1]   
1    [C(=O)c(cc1)ccc1C(=O), Oc1c(C)cc(cc1(C))C(C)(C...     [1, 1]   
2    [C(=O)C(=O), Oc1ccc(cc1)C(C)(c2ccccc2)c3ccc(cc...     [1, 1]   
3                               [C(=O)CC(=O), OCC=CCO]     [1, 1]   
4                           [C(=O)C(=O), OCCCCCCCCCCO]     [1, 1]   
..                                                 ...        ...   
651                 [C(=O)C=Cc(cccc1)c1O, C(=O)CCCCCO]     [2, 1]   
652                  [C(=O)C=Cc(cccc1)c1O, C(=O)CCCCO]     [2, 1]   
653                  [C(=O)CCCCO, C(=O)C=Cc(cccc1)c1O]     [2, 1]   
654              [C(=O)CCCCC(CCCC)O, C(=O)CCCC(CCCC)O]     [2, 1]   
655                       [C(=O)COC(=O)CO, C(=O)CCCCO]     [2, 1]   

     Biodegradability  
0                   0  
1                   0  
2                   1  
3                   1  
4                   1  
..                ...  
651

In [46]:
def get_smiles_and_ratios_list_dataframe(dataframe):
    names=dataframe['Polymer_Name_P'].to_list()
    ratio1=dataframe['Ratio 1'].to_list()
    ratio2=dataframe['Ratio 2'].to_list()
    ratio3=dataframe['Ratio 3'].to_list()
    smiles=[]
    ratios=[]
    monomers=pd.read_csv('./Monomer Info_DMTDMI.csv',header=0)
    for i in range(len(names)):
        name=names[i]
        smile=[]
        smile+=[str(monomers.loc[monomers['Polymer Name Abbr.']==name[0:3],'SMILES_Monomers'].item())]
        smile+=[str(monomers.loc[monomers['Polymer Name Abbr.']==name[6:9],'SMILES_Monomers'].item())]
        smile+=[str(monomers.loc[monomers['Polymer Name Abbr.']==name[12:15],'SMILES_Monomers'].item())]
        #print(smile)
        smiles+=[smile]
        #print(smiles)
        ratios+=[[ratio1[i]/2,ratio2[i]/2,ratio3[i]/2]]
    df=pd.DataFrame({'SMILES LIST':smiles,'Ratio List':ratios,'Biodegradability':dataframe['Biodegradable']}).reset_index(drop=True)
    return df

In [153]:
#testing some things
print(ter_all)
ter_all_corr=ter_all.replace(["Y","N"],[1,0])
ter_all_corr=ter_all_corr.reset_index(drop=True)
df1=ter_all_corr
df1=get_smiles_and_ratios_list_dataframe(df1)
print(df1["SMILES LIST"][1])
mask = df1["SMILES LIST"].apply(lambda x: any('COC' in item for item in x))
print(mask)

print('original', df1)
print('mask', df1[mask])
print('inverse mask',df1[~mask])

          Polymer_Name_P Biodegradable  Ratio 1  Ratio 2  Ratio 3
0    ADA25_DMI25_BND50_P             Y      1.4      0.6      2.0
1    ADA25_DMI25_BTD50_P             Y      0.9      1.1      2.0
2    ADA25_DMT25_BPA50_P             N      1.8      0.2      2.0
3    ADA25_DMT25_DPD50_P             N      1.1      0.9      2.0
4    ADA25_DMT25_HND50_P             Y      1.2      0.8      2.0
..                   ...           ...      ...      ...      ...
230  DGA25_CCD25_GGE50_P             Y      1.0      1.0      2.0
231  EBA25_AZA25_BPA50_P             Y      0.9      1.1      2.0
232  GLA25_SBA25_BPA50_P             Y      1.1      0.9      2.0
233  EBA25_SBA25_BPA50_P             Y      0.9      1.1      2.0
234  GLA25_SUA25_BPA50_P             N      1.7      0.3      2.0

[300 rows x 5 columns]
['C(=O)CCCCC(=O)', 'C(=O)c(ccc1)cc1C(=O)', 'OC(C)C(C)O']
0      False
1      False
2      False
3      False
4      False
       ...  
295     True
296     True
297    False
298     Tr

/var/folders/t5/wsvlnfc52kd7dmby8bpwq66r0000gn/T/ipykernel_72556/4030480842.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ter_all_corr=ter_all.replace(["Y","N"],[1,0])


In [92]:
#write an overall dataframe for the terpolymers as smile strings
ter_dataframe = get_smiles_and_ratios_list_dataframe(ter_all_corr)
print(ter_dataframe)

                                           SMILES LIST         Ratio List  \
0       [C(=O)CCCCC(=O), C(=O)c(ccc1)cc1C(=O), OCCCCO]    [0.7, 0.3, 1.0]   
1    [C(=O)CCCCC(=O), C(=O)c(ccc1)cc1C(=O), OC(C)C(...  [0.45, 0.55, 1.0]   
2    [C(=O)CCCCC(=O), C(=O)c(cc1)ccc1C(=O), Oc1ccc(...    [0.9, 0.1, 1.0]   
3    [C(=O)CCCCC(=O), C(=O)c(cc1)ccc1C(=O), OCC(CC)...  [0.55, 0.45, 1.0]   
4     [C(=O)CCCCC(=O), C(=O)c(cc1)ccc1C(=O), OCCCCCCO]    [0.6, 0.4, 1.0]   
..                                                 ...                ...   
295  [C(=O)COCC(=O), C(=O)C(CC1)CCC1C(=O), OCC(COc1...    [0.5, 0.5, 1.0]   
296  [C(=O)COCCOCC(=O), C(=O)CCCCCCCC(=O), Oc1ccc(c...  [0.45, 0.55, 1.0]   
297  [C(=O)CCCC(=O), C(=O)CCCCCCCCC(=O), Oc1ccc(cc1...  [0.55, 0.45, 1.0]   
298  [C(=O)COCCOCC(=O), C(=O)CCCCCCCCC(=O), Oc1ccc(...  [0.45, 0.55, 1.0]   
299  [C(=O)CCCC(=O), C(=O)CCCCCCC(=O), Oc1ccc(cc1)C...  [0.85, 0.15, 1.0]   

     Biodegradability  
0                   1  
1                   1  
2  

In [147]:
#write landscape dataframe (nothing removed, will remove relevant polymers in each iteration

landscape_dataframe = pd.DataFrame({'SMILES LIST': os0['SMILES_List'], 'Ratio List': os0['Ratio_List'],'Biodegradability': os0['Biodegradability']}).reset_index(drop=True)
landscape_dataframe['SMILES LIST']=landscape_dataframe['SMILES LIST'].apply(ast.literal_eval)
landscape_dataframe.to_csv('./Chemical_extensibility/landscape_basis.csv',header=True)
print(landscape_dataframe)

                                           SMILES LIST Ratio List  \
0                   [C(=O)c(cc1)ccc1C(=O), OCCCCCCCCO]     [1, 1]   
1    [C(=O)c(cc1)ccc1C(=O), Oc1c(C)cc(cc1(C))C(C)(C...     [1, 1]   
2    [C(=O)C(=O), Oc1ccc(cc1)C(C)(c2ccccc2)c3ccc(cc...     [1, 1]   
3                               [C(=O)CC(=O), OCC=CCO]     [1, 1]   
4                           [C(=O)C(=O), OCCCCCCCCCCO]     [1, 1]   
..                                                 ...        ...   
104             [C(=O)COC(=O)CO, C(=O)CCCCCCCCCCCCCCO]     [2, 1]   
105              [C(=O)COC(=O)CO, C(=O)C=Cc(cccc1)c1O]     [2, 1]   
106           [C(=O)C=Cc(cccc1)c1O, C(=O)CCCCC(CCCC)O]     [2, 1]   
107          [C(=O)C=Cc(cccc1)c1O, C(=O)C=CCC(CCCCC)O]     [2, 1]   
108                         [C(=O)COC(=O)CO, C(=O)CCO]     [2, 1]   

     Biodegradability  
0                   0  
1                   0  
2                   1  
3                   1  
4                   1  
..                ...  
104

In [149]:
#put all the data into one large dataset to pull from
dataset = pd.concat([orig_dataframe,ter_dataframe]).reset_index(drop=True)
print(dataset)
dataset.to_csv('./Chemical_extensibility/polymers_combined.csv',header=True)

                                           SMILES LIST         Ratio List  \
0                   [C(=O)c(cc1)ccc1C(=O), OCCCCCCCCO]             [1, 1]   
1    [C(=O)c(cc1)ccc1C(=O), Oc1c(C)cc(cc1(C))C(C)(C...             [1, 1]   
2    [C(=O)C(=O), Oc1ccc(cc1)C(C)(c2ccccc2)c3ccc(cc...             [1, 1]   
3                               [C(=O)CC(=O), OCC=CCO]             [1, 1]   
4                           [C(=O)C(=O), OCCCCCCCCCCO]             [1, 1]   
..                                                 ...                ...   
951  [C(=O)COCC(=O), C(=O)C(CC1)CCC1C(=O), OCC(COc1...    [0.5, 0.5, 1.0]   
952  [C(=O)COCCOCC(=O), C(=O)CCCCCCCC(=O), Oc1ccc(c...  [0.45, 0.55, 1.0]   
953  [C(=O)CCCC(=O), C(=O)CCCCCCCCC(=O), Oc1ccc(cc1...  [0.55, 0.45, 1.0]   
954  [C(=O)COCCOCC(=O), C(=O)CCCCCCCCC(=O), Oc1ccc(...  [0.45, 0.55, 1.0]   
955  [C(=O)CCCC(=O), C(=O)CCCCCCC(=O), Oc1ccc(cc1)C...  [0.85, 0.15, 1.0]   

     Biodegradability  
0                   0  
1                   0  
2  

In [151]:
#write function that will generate mask for you
def mask_writer(dataframe, str):
    mask = dataframe['SMILES LIST'].apply(lambda x: any(str in item for item in x))
    return mask

In [161]:
#write the list of masks we are interested in
#[backbone ether, sulfur, aromatic, aliphatic rings]
mask_str_list=['COC','S','c','C1']
landscapes=[] #list of landscapes with the polymers containing chemical feature of interest removed
datasets=[] #training data with the polymers containing chemical feature of interest removed
test = [] #test set of data with the polymers containing chemical feature of interest
#mask = landscape_dataframe['SMILES LIST'].apply(lambda x: any('COC' in item for item in x))
#print(mask)
for i in range(len(mask_str_list)):
    if i == 0:
        mask1 = mask_writer(dataset,mask_str_list[i]) #removes the generic ethers
        mask2 = mask_writer(dataset, 'OCC(C)OCC(C)O') #removes dipropylene glycol which doesn't get caught
        data_mask = mask1 | mask2
    else:
        data_mask=mask_writer(dataset,mask_str_list[i])
    #print(data_mask)
    poly=dataset[~data_mask]
    #print(poly)
    land=poly.loc[poly.index<109]
    land=land.reset_index(drop=True)
    poly=dataset[~data_mask].reset_index(drop=True)
    print(land)
    test_set=dataset[data_mask].reset_index(drop=True)
    landscapes+=[land]
    datasets+=[poly] #all the polymers that don't fit the mask criteria
    test += [test_set] #all the polymers that do meet the mask criteria
    poly.to_csv('./Chemical_extensibility/polymers_train_'+str(mask_str_list[i])+'.csv',header=True)
    test_set.to_csv('./Chemical_extensibility/polymers_test_'+str(mask_str_list[i])+'.csv',header=True)
    land.to_csv('./Chemical_extensibility/polymers_land_'+str(mask_str_list[i])+'.csv',header=True)
   # calculate the EMD set for each of the data subsets
    arrays=get_EMD_array([poly,test_set],land)
    np.save('./Chemical_extensibility/train_'+str(mask_str_list[i]),arrays[0],allow_pickle=False)
    np.save('./Chemical_extensibility/test_'+str(mask_str_list[i]),arrays[1],allow_pickle=False)

                                          SMILES LIST Ratio List  \
0                  [C(=O)c(cc1)ccc1C(=O), OCCCCCCCCO]     [1, 1]   
1   [C(=O)c(cc1)ccc1C(=O), Oc1c(C)cc(cc1(C))C(C)(C...     [1, 1]   
2   [C(=O)C(=O), Oc1ccc(cc1)C(C)(c2ccccc2)c3ccc(cc...     [1, 1]   
3                              [C(=O)CC(=O), OCC=CCO]     [1, 1]   
4                          [C(=O)C(=O), OCCCCCCCCCCO]     [1, 1]   
..                                                ...        ...   
89                             [C(=O)CCCCO, C(=O)CCO]     [2, 1]   
90         [C(=O)C=Cc(cccc1)c1O, C(=O)C=Cc(cccc1)c1O]     [2, 1]   
91         [C(=O)C(C)OC(=O)C(C)O, C(=O)CCCC(CCCCCC)O]     [2, 1]   
92           [C(=O)C=Cc(cccc1)c1O, C(=O)CCCCC(CCCC)O]     [2, 1]   
93          [C(=O)C=Cc(cccc1)c1O, C(=O)C=CCC(CCCCC)O]     [2, 1]   

    Biodegradability  
0                  0  
1                  0  
2                  1  
3                  1  
4                  1  
..               ...  
89                 1  

In [162]:
#load train and test X and make np ys
train_c_X=np.load('./Chemical_extensibility/train_c.npy')
train_C1_X=np.load('./Chemical_extensibility/train_C1.npy')
train_COC_X=np.load('./Chemical_extensibility/train_COC.npy')
train_S_X=np.load('./Chemical_extensibility/train_S.npy')
test_c_X=np.load('./Chemical_extensibility/test_c.npy')
test_C1_X=np.load('./Chemical_extensibility/test_C1.npy')
test_COC_X=np.load('./Chemical_extensibility/test_COC.npy')
test_S_X=np.load('./Chemical_extensibility/test_S.npy')
#same order as the datasets and test set lists
train_Xs=[train_COC_X,train_S_X,train_c_X,train_C1_X]
test_Xs=[test_COC_X,test_S_X,test_c_X,test_C1_X]

#write the ys for the different sets
train_ys=[]
test_ys=[]
for i in range(4):
    trainy=datasets[i]['Biodegradability'].to_numpy()
    train_ys+=[trainy]
    testy=test[i]['Biodegradability'].to_numpy()
    test_ys+=[testy]
print(train_ys)




[array([0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1,
       0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0,
       1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0,
       0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1,
       0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0,
       0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0,
       0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0,
       1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1,
       0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1,
       1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0

In [174]:
#use the optimized hyperparameters for the joint original library and copolymer library
param=[32,32,64]
scores=[]
mask_str_list=['COC','S','c','C1']
#def train_rf(C,xs,ys,name):
    #C should be [n_estimators, max_depth,max_leaf_nnodes]
    #xs and ys should be list of the training data (no validation set, test set should be left out)
    # X = concatenate_indices(xs, None)
    # Y = concatenate_indices(ys, None)
    #returns the trained model (also saves it)
for i in range(4):
    #train the model
    model=train_rf(param,train_Xs[i],train_ys[i],mask_str_list[i])
    eval = model.score(test_Xs[i],test_ys[i])
    #create mask for splitting original and copolymer test sets
    mask_lib=test[i]['SMILES LIST'].str.len() == 2
    orig_test_Xs=test_Xs[i][mask_lib.values]
    orig_test_ys=test_ys[i][mask_lib.values]
    cop_test_Xs=test_Xs[i][~mask_lib.values]
    cop_test_ys=test_ys[i][~mask_lib.values]
    evalorig=model.score(orig_test_Xs,orig_test_ys)
    evalcop=model.score(cop_test_Xs,cop_test_ys)
    scores+=[[evalorig,evalcop,eval]]

print(scores)
scores = np.array(scores)
np.save('./Chemical_extensibility/test_scores',scores,allow_pickle=False)

X (731, 94)
Y (731,)
X (830, 98)
Y (830,)
X (596, 62)
Y (596,)
X (772, 92)
Y (772,)
[[0.6095238095238096, 0.825, 0.7244444444444444], [0.5977011494252874, 0.9230769230769231, 0.6984126984126984], [0.4749034749034749, 0.7029702970297029, 0.5388888888888889], [0.5700934579439252, 0.7532467532467533, 0.6467391304347826]]


In [170]:
#sort out how to split up the test set based on original or copolymer to implement above
print(test[0])
testing=test[0]
mask= test[0]['SMILES LIST'].str.len() ==2
print(test[0][mask])
print(test[0][~mask])

                                           SMILES LIST         Ratio List  \
0          [C(=O)c(cc1)ccc1C(=O), OC(C)COC(C)COC(C)CO]             [1, 1]   
1                         [C(=O)COCC(=O), OCCCCCCCCCO]             [1, 1]   
2       [C(=O)COCC(=O), Oc1ccc(cc1)C(C)(C)c2ccc(cc2)O]             [1, 1]   
3                            [C(=O)COCC(=O), OCC(C)CO]             [1, 1]   
4                           [C(=O)CCCCCC(=O), OCCOCCO]             [1, 1]   
..                                                 ...                ...   
220     [C(=O)COCC(=O), C(=O)CCCCCCCC(=O), OCCCCCCCCO]    [0.5, 0.5, 1.0]   
221   [C(=O)CCCCCCC(=O), C(=O)COCCOCC(=O), OCCCCCCCCO]  [0.55, 0.45, 1.0]   
222  [C(=O)COCC(=O), C(=O)C(CC1)CCC1C(=O), OCC(COc1...    [0.5, 0.5, 1.0]   
223  [C(=O)COCCOCC(=O), C(=O)CCCCCCCC(=O), Oc1ccc(c...  [0.45, 0.55, 1.0]   
224  [C(=O)COCCOCC(=O), C(=O)CCCCCCCCC(=O), Oc1ccc(...  [0.45, 0.55, 1.0]   

     Biodegradability  
0                   1  
1                   1  
2  